# Vietnam Stock Market — ML direction forecast (SMA Project)

End-to-end notebook for the ML component of the Stock Market Analytics project.

**Question:** can lecture-style ML models (Decision Tree / Random Forest), fed only
price-derived features known at week *t*, predict whether a Vietnamese stock will be
higher 8 weeks later?

**Setup (point-in-time, no lookahead):**
- Data: Simplize OHLCV (matched-order volume only), weekly bars (close of every 5th session).
- Universe: 254 tickers (>= 5 years of data, last price 2,000-500,000 VND, median daily volume >= 50k shares).
- Features at week t: returns 4/12/26/52w, weekly RSI14, distance to SMA20/SMA50, distance to 52-week high, volume ratio vs 20w mean.
- Target: 1 if close[t+8] > close[t]. Split by time: train <= 2021, test >= 2022.
- Data files are not committed (course rules) - set SMA_DATA_DIR to a local bars store (see data/README.md).

In [1]:
import os
os.environ.setdefault("SMA_DATA_DIR", "data/bars")
import pandas as pd
from ml_model import build_panel, FEATS, TRAIN_END

panel = build_panel()
print(f"Panel: {len(panel):,} ticker-weeks, {panel['date'].min().date()} -> {panel['date'].max().date()}")
panel.head()

Universe: 254 ma giu lai, 1266 ma loai (thieu du lieu / ngoai LIQ)
Panel: 158,925 ticker-weeks, 2002-04-05 -> 2026-09-25


,date,ret4,ret12,ret26,ret52,rsi14,dist_sma20,dist_sma50,dist_52w_high,vol_ratio,fwd8,ticker,y
0,2011-08-08 07:00:00+07:00,-0.151515,-0.289340,-0.596542,-0.689579,29.981771,-0.270643,-0.569390,-0.845133,0.617092,0.234940,AAA,1
1,2011-08-15 07:00:00+07:00,-0.138365,-0.121795,-0.600583,-0.695556,29.539134,-0.262450,-0.570506,-0.848451,0.371521,0.254287,AAA,1
2,2011-08-22 07:00:00+07:00,0.033113,0.075862,-0.547826,-0.688000,35.985106,-0.136929,-0.501916,-0.827434,1.452332,0.013670,AAA,1
3,2011-08-29 07:00:00+07:00,0.143885,-0.091429,-0.457338,-0.641892,36.965657,-0.096334,-0.479064,-0.824115,1.570660,0.014435,AAA,1
4,2011-09-06 07:00:00+07:00,0.100000,-0.241379,-0.442029,-0.653933,35.976553,-0.101517,-0.473144,-0.829646,1.721316,0.081599,AAA,1


## EDA — panel summary, target distribution, feature correlations


In [2]:
# Summary statistics of the 9 features and the 8-week forward return
panel[FEATS + ["fwd8"]].describe().T[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]


,count,mean,std,min,25%,50%,75%,max
ret4,158925.0,0.013995,0.143801,-0.746595,-0.057971,0.000000,0.071429,3.888889
ret12,158925.0,0.044322,0.277642,-0.930565,-0.104478,0.000000,0.142857,7.327273
ret26,158925.0,0.104259,0.468511,-0.931104,-0.154030,0.014152,0.250000,11.200000
ret52,158925.0,0.234435,0.826100,-0.932143,-0.198766,0.052718,0.423684,26.865329
rsi14,158925.0,50.966650,13.122583,8.306919,41.783656,50.041762,59.276006,98.100303
dist_sma20,158925.0,0.017799,0.178232,-0.882782,-0.076479,-0.000917,0.093325,2.946396
dist_sma50,158925.0,0.049388,0.306847,-0.896220,-0.124810,0.010379,0.179337,5.188406
dist_52w_high,158925.0,-0.232609,0.189952,-0.946176,-0.340906,-0.193966,-0.082353,0.000000
vol_ratio,158925.0,1.072213,0.959134,0.000000,0.531422,0.847712,1.317900,19.946019
fwd8,158925.0,0.028669,0.214627,-0.894265,-0.084112,0.000000,0.109091,5.000000


In [3]:
# Target distribution: share of ticker-weeks that rose over the next 8 weeks
print(f"Up-rate (y=1): {panel['y'].mean():.3f}  |  fwd8 median: {panel['fwd8'].median():+.3f}  |  fwd8 mean: {panel['fwd8'].mean():+.3f}")
panel["y"].value_counts(normalize=True).rename("share")


Up-rate (y=1): 0.494  |  fwd8 median: +0.000  |  fwd8 mean: +0.029


y
0    0.505918
1    0.494082
Name: share, dtype: float64

In [4]:
# Correlation of each feature with the forward return and with the target
corr = panel[FEATS + ["fwd8", "y"]].corr()[["fwd8", "y"]].drop(index=["fwd8", "y"])
corr.sort_values("y", ascending=False)


,fwd8,y
rsi14,0.058877,0.039601
dist_sma50,0.038922,0.023984
dist_52w_high,0.018616,0.023889
vol_ratio,0.042670,0.023406
ret26,0.029471,0.016769
dist_sma20,0.042470,0.014779
ret4,0.032087,0.009536
ret52,0.006390,0.008277
ret12,0.034004,0.007925


In [5]:
tr = panel[panel["date"] <= TRAIN_END]
te = panel[panel["date"] > TRAIN_END]
print(f"Train (<=2021): {len(tr):,} rows | Test (>=2022): {len(te):,} rows")
print(f"Up-rate 8w - train: {tr['y'].mean():.3f} | test: {te['y'].mean():.3f}  (test period leans bearish)")
Xtr, ytr = tr[FEATS], tr["y"]
Xte, yte = te[FEATS], te["y"]

Train (<=2021): 99,118 rows | Test (>=2022): 59,807 rows
Up-rate 8w - train: 0.513 | test: 0.463  (test period leans bearish)


## Models vs baselines

Baselines matter: in the test period only 46.4% of ticker-weeks rose, so "always predict
down" scores 0.536 accuracy and a coin flip has AUC 0.500. A model only counts if it
beats those numbers out of sample.

In [6]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from ml_model import evaluate

results = []
maj = int(ytr.mean() >= 0.5)
base_pred = np.full(len(yte), maj)
results.append({"model": "Majority (blind)", "acc": round(float((base_pred == yte).mean()), 4),
                "auc": 0.5, "f1": round(float(f1_score(yte, base_pred, zero_division=0)), 4)})
print(f"{'Majority (blind)':28s} acc={results[0]['acc']:.3f}  AUC=0.500  F1={results[0]['f1']:.3f}")
print(f"{'Always-down baseline':28s} acc={(yte == 0).mean():.3f}")

for depth in [3, 5, 10, None]:
    results.append(evaluate(f"DecisionTree depth={depth}",
                            DecisionTreeClassifier(max_depth=depth, random_state=0),
                            Xtr, ytr, Xte, yte))
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=0)
results.append(evaluate("RandomForest n=200", rf, Xtr, ytr, Xte, yte))
pd.DataFrame(results)

Majority (blind)             acc=0.463  AUC=0.500  F1=0.633
Always-down baseline         acc=0.537


DecisionTree depth=3         acc=0.486  AUC=0.496  F1=0.502


DecisionTree depth=5         acc=0.482  AUC=0.492  F1=0.506


DecisionTree depth=10        acc=0.502  AUC=0.504  F1=0.466


DecisionTree depth=None      acc=0.504  AUC=0.504  F1=0.488


RandomForest n=200           acc=0.507  AUC=0.516  F1=0.505


,model,acc,auc,f1
0,Majority (blind),0.4628,0.5000,0.6327
1,DecisionTree depth=3,0.4863,0.4961,0.5019
2,DecisionTree depth=5,0.4819,0.4923,0.5056
3,DecisionTree depth=10,0.5016,0.5036,0.4658
4,DecisionTree depth=None,0.5038,0.5043,0.4881
5,RandomForest n=200,0.5071,0.5161,0.5054


In [7]:
imp = pd.Series(rf.feature_importances_, index=FEATS).sort_values(ascending=False)
print("Feature importance (RandomForest):")
for k, v in imp.items():
    print(f"  {k:14s} {v:.3f}")

Feature importance (RandomForest):
  dist_sma50     0.119
  ret52          0.117
  ret26          0.116
  rsi14          0.111
  dist_52w_high  0.109
  dist_sma20     0.109
  ret12          0.108
  vol_ratio      0.107
  ret4           0.104


## Regime variant (already run - see results/sma-ml-prototype-regime.md)

Adding the strategy's own regime definition as features (equal-weight index vs its
40-week SMA: `regime_bull`, `dist_eqw_sma40`) made things **worse**, not better:
RandomForest AUC fell to **0.482** (below a coin flip), Decision Trees dropped to
0.469-0.498. Split by regime, the model is below 0.5 in both bull weeks (AUC 0.469)
and bear weeks (0.497) - so the weak base result is not an artefact of a bearish
test period. Reproduce with `python ml_model_regime.py`.

## Conclusion (reported as-is)

Price-only features do **not** predict the 8-week direction on Vietnamese data out
of sample: the best model (RandomForest, AUC 0.516 in the standalone run) barely
matches a coin flip and loses to the always-down baseline on accuracy, and feature
importances are flat. This negative result is the finding. The trading system in
this project therefore stays rule-based (Strategy v1.2, see STRATEGY.md and
results/backtest_summary.md) and does not use ML forecasts as signals.

## End-to-end workflow: simulate + latest trades

The full automation chain of this project is: **download** (the live pipeline fetches Simplize OHLCV daily into the local store — in this repo the store is read via `SMA_DATA_DIR`) -> **transform** (`build_panel`, point-in-time features) -> **predict** (RandomForest probabilities) -> **simulate** (`trading_simulation.simulate`: two vector portfolios vs the equal-weight benchmark) -> **latest trades** (top-10 signals of the panel's last week + the regime state). The rule-based strategy v1.2 runs the same chain in production via `strategy/paper_trade.py` (plan Saturday, execute Monday).


In [8]:
import trading_simulation as TS

te_sim = te.copy()
te_sim["proba"] = rf.predict_proba(Xte)[:, 1]
sim_summary, sim_trades, sim_periods = TS.simulate(te_sim)
print(f"Sim 1 top-10          : total {sim_summary['sim1_total']:+.1%} | win {sim_summary['sim1_win']:.0%} over {sim_summary['n_periods']} periods")
print(f"Sim 2 top-10 & p>0.55 : total {sim_summary['sim2_total']:+.1%} | win {sim_summary['sim2_win']:.0%} (invested {sim_summary['sim2_invested_periods']}/{sim_summary['n_periods']})")
print(f"Benchmark EQW         : total {sim_summary['bench_total']:+.1%} | win {sim_summary['bench_win']:.0%}")
sim_periods.head(10)


trading_simulation.py:55: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  t["week"] = t["date"].dt.to_period("W")


Sim 1 top-10          : total +156.5% | win 74% over 31 periods
Sim 2 top-10 & p>0.55 : total +149.7% | win 74% (invested 31/31)
Benchmark EQW         : total -2.2% | win 52%


,date,sim1_top10,sim2_thr,bench_eqw,n_thr
0,2021-12-27/2022-01-02,0.043101,0.015228,0.043101,1
1,2022-02-28/2022-03-06,-0.205537,-0.205537,-0.153479,10
2,2022-04-25/2022-05-01,-0.303487,-0.303487,-0.184428,10
3,2022-06-20/2022-06-26,0.112149,0.112149,0.149857,10
4,2022-08-15/2022-08-21,-0.150843,-0.150843,-0.206886,10
5,2022-10-10/2022-10-16,0.115359,0.115359,-0.046293,10
6,2022-12-05/2022-12-11,0.048187,0.048187,0.025438,10
7,2023-01-30/2023-02-05,0.002883,0.002883,-0.027726,10
8,2023-03-27/2023-04-02,0.230765,0.230765,0.158584,10
9,2023-05-22/2023-05-28,0.093345,0.093345,0.158393,10


In [9]:
# Latest new trades: top-10 tickers by predicted up-probability at the
# panel's last week, plus the market regime that week (EQW vs SMA40w)
last_date, latest_top, regime = TS.latest_signals(panel, rf)
print(f"Latest panel week: {last_date.date()} | regime: {'BULL' if regime['bull'] else 'BEAR'} (EQW vs SMA40w {regime['dist_eqw_sma40']:+.2%})")
latest_top


trading_simulation.py:101: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  pp["week"] = pp["date"].dt.to_period("W")


Universe: 254 ma giu lai, 1266 ma loai
EQW: 1259 tuan lich, bull share toan panel: 0.687 (bo 0 dong 40 tuan dau chua co SMA40)
Latest panel week: 2026-09-25 | regime: BEAR (EQW vs SMA40w -13.08%)


ml_model_regime.py:83: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  p["week"] = p["date"].dt.to_period("W")
ml_model_regime.py:92: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  panel["week"] = panel["date"].dt.to_period("W")


,ticker,proba
106446,PVC,0.838683
6694,APH,0.728502
83715,MSB,0.719961
31712,DDV,0.713047
109551,PVP,0.701591
59274,HHP,0.697655
99172,PET,0.694966
85344,MSR,0.685344
113682,QTP,0.667794
35462,DPG,0.644626
